
<img src="../img/GTK_Logo_Social_Icon.jpg" width=175 align="right" />

# Worksheet 7.1 Deep Learning: Convolutional Neural Networks

This notebook shows three commonly used neural network architectures to train a model that can classify images of fingerprints.

Data source: JinZhuXing [Fingerprint Dataset for FVC2000_DB4_B in Kaggle](https://www.kaggle.com/peace1019/fingerprint-dataset-for-fvc2000-db4-b)

Useful Notebooks: 

[https://www.kaggle.com/code/peace1019/train-and-eval-20200410](https://www.kaggle.com/code/peace1019/train-and-eval-20200410)

[https://www.kaggle.com/code/sayeefmahmud/fingerprint-detector](https://www.kaggle.com/code/sayeefmahmud/fingerprint-detector)

Libraries for this worksheet:
- [Keras](https://keras.io/) is used as high-level API for [tensorflow](https://www.tensorflow.org/) backend
- PIL
- shutil (built in python library)
- collections
- scikit-learn
- pydot (graphViz to print out the model layers)
- numpy
- matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model

from keras.preprocessing import image
from keras.utils import to_categorical
from keras.applications import VGG16
from keras import layers, models,  regularizers, optimizers, callbacks
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras.models import Sequential

#import pydot
from sklearn.model_selection import train_test_split
import random
from PIL import Image
from collections import defaultdict
import shutil
import os

print("Version: ", tf.__version__)

In [ ]:
import sys
sys.version

# Load Data

1. Go into the ```../data/fingerprints/``` directory and extract the `.zip` file. This will result in a `train_data` directory that has a bunch of `.bmp` images (our raw data).  

The  .bmp files are named with a 4-digit ID for the person, followed by a two-digit sequence number pertaining to (likely) the finger number, though we don't know for sure so we will just treat it as a sequential ID. 
```
00001_04.bmp
00001_06.bmp
```


In [ ]:
DATA_HOME = '../data/fingerprints'

# Path to dataset folder
TRAIN_DATA_RAW_PATH = os.path.join(DATA_HOME, "train_data")

In [ ]:
# Dictionary to store the count of fingerprints per person
person_fingerprints = defaultdict(int)

# Iterate over files
for filename in os.listdir(TRAIN_DATA_RAW_PATH):
    if filename.endswith(".bmp"): 
        person_id = filename.split("_")[0]  # Extract the person ID 
        person_fingerprints[person_id] += 1

### Visualize Data

In [ ]:
# Randomly sample and plot one of our raw data files
sample_images = []
for file_id in random.sample(os.listdir(TRAIN_DATA_RAW_PATH), 1):  
    img_path =  os.path.join(TRAIN_DATA_RAW_PATH, file_id) 

# Plot the fingerprint
plt.figure(figsize=(10, 10))
img = Image.open(img_path)
plt.subplot(3, 2, 1)
plt.imshow(img) #, cmap="gray")  # change to grayscale image, since that's what it is irl
plt.title(f"File ID: {file_id}")
plt.axis("off")
print(f"Image: {img_path}, Dimensions: {img.size}")  # Print dimensions

plt.tight_layout()
plt.show()

## Load Data Part 2
In order to train the model, we will need to separate the raw files into a train/test/validation split. 
- The training data in **TRAIN_DATA_RAW_PATH**, will need to be organized by person in order to stratify our splits. 

In [ ]:
# Organize fingerprint files by person
fingerprints_by_person = {}

# Create a dictionary that has each person's ID as the keys and a list of that person's filenames as the value
for filename in os.listdir(TRAIN_DATA_RAW_PATH):
    if filename.endswith(".bmp"):
        person_id = filename.split("_")[0]  # Extract the person ID
        if person_id not in fingerprints_by_person:
            fingerprints_by_person[person_id] = []
        fingerprints_by_person[person_id].append(filename)

In [ ]:
# check our dict for one person
fingerprints_by_person['00001'][1:5]

### Exercise 1a: Count the number of different people in this dataset, and their fingerprints

In [ ]:
# Total number of unique person's
num_persons = # Your code here...


## Train/Test/Validation split
We see that there are usually more than 10 files per person, so it's probably safe to assume that the sequence number does not map to anything.

Next we make the train/test/val split. When doing a multi-class problem like this, we want to make sure we get some data from each class into the train/test and validation datasets. So we need to iterate through each person in the dict and randomly pull out the following ratios:
```
train=70%
validation=15%
test=15%
```

In [ ]:
# Output directories 
output_dir = os.path.normpath(os.path.join(DATA_HOME, "split_dataset"))
train_dir = os.path.normpath(os.path.join(output_dir, "train"))
val_dir = os.path.normpath(os.path.join(output_dir, "validation"))
test_dir = os.path.normpath(os.path.join(output_dir, "test"))

# Create directories if they don't exist
    
# Split data into train/val/test for each person

# Copy files to the corresponding directories


### Visualize Data
Let's look at a few samples for 3 people. 

In [ ]:
# Randomly sample some person IDs and their fingerprints
sample_images = []
# Your code here...

# Plot the samples
plt.figure(figsize=(10, 10))
for i, (person_id, img_path) in enumerate(sample_images, 1):
    img = Image.open(img_path)
    plt.subplot(3, 2, i)
    plt.imshow(img, cmap="gray")  # raw data is grayscale images
    plt.title(f"Person ID: {person_id}")
    plt.axis("off")
    print(f"Image: {img_path}, Dimensions: {img.size}")  # Print dimensions

plt.tight_layout()
plt.show()

# Exercise: Train a CNN from scratch to correctly identify the person a fingerprint belongs to
We want to train a model that can classify a fingerprint image as the correct person. This is a multi-class problem (as opposed to a binary classification problem e.g. Hotdog/not hotdog). Keep in mind that we will need to evaluate this model a bit differently than we do for a binary classification problem.

## Data Pre-processing
- resize images to be the same *(160, 160)*
- normalize
- convert to grayscale
- save as a numpy array for input into the CNN

In [ ]:
def load_and_preprocess_images(image_paths):
    '''
     Load and preprocess grayscale fingerprint images for input into a CNN model. 
     '''
    images = []
    labels = []
    for person_id in os.listdir(image_paths):
        person_path = os.path.join(image_paths, person_id)
        if os.path.isdir(person_path):
            for img_file in os.listdir(person_path):
                img_path = os.path.join(person_path, img_file)
                # Load the image
                img = image.load_img(img_path, target_size=(160, 160), color_mode="grayscale")
                # Convert the image to a NumPy array
                img_array = image.img_to_array(img)
                images.append(img_array)
                labels.append(int(person_id))  

    return np.array(images), np.array(labels)

# Load training and validation data

## Create a CNN model architecture

In [ ]:
final_Dense_units = 10
model_name = 'SubjectIDClassifier'

model_tb = Sequential(name=model_name)
# Your code here...

# Complete with Adam optimizer and entropy cost


## Set Hyperparameters:
- Set the **epochs**
- Set the **batch_size**

### Callbacks
- Callbacks in Keras are objects that are called at different points during training (at the start of an epoch, at the end of a batch, at the end of an epoch, etc.). They can be used to implement certain behaviors, such as:

- Doing validation at different points during training (beyond the built-in per-epoch validation)
- Checkpointing the model at regular intervals or when it exceeds a certain accuracy threshold
- Changing the learning rate of the model when training seems to be plateauing
- Doing fine-tuning of the top layers when training seems to be plateauing
- Sending email or instant message notifications when training ends or where a certain performance threshold is exceeded
- Etc.

Callbacks can be passed as a list to your call to fit():


In [ ]:
epochs = 20
batch_size = 64

CallBack = [
        callbacks.EarlyStopping(
            # Stop training when `val_loss` is no longer improving

            # "no longer improving" being defined as "no better than 1e-2 less"

            # "no longer improving" being further defined as "for at least 2 epochs"


         callbacks.ModelCheckpoint("save_at_{epoch}.keras"),
]
            
           
        #callbacks.TensorBoard(log_dir="./log_dir/"+model_name)

## Train (fit) the model 
**Fit** the model to the training data that you just prepared and use the validation data to evaluate the model.
- HINT: see the **validation_data** parameter for the *.fit* method)

In [ ]:
# Your code here...

## Evaluation
Evaluate the model on the test set. 

1. What is the accuracy of our model on the test set?
2. Is this much lower/higher than what was happening above on the validation set?
3. Train the model again and adjust some of the parameters of the model to try to improve the performance of the model. 

In [ ]:
# Evaluate on the test set
test_loss, test_acc = # Your code here...
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_acc}")

## Saving your model (optional)
Saving a model is straightforward, it can be done with the *pickle* library, or the *h5* library, or the keras native format. 

In [ ]:
# Your code here...

# Exercise: Fingerprint Authentication
Create an additional function that will take in a raw .bmp image of a fingerprint, predict the owner's ID, and validate whether the model has correctly predicted the owner.

In [ ]:
def authenticate_fingerprint(one_image, model):
    
    # Load the image
    plt.figure(figsize=(10, 10))
   # img = Image.open( image_path)
    plt.subplot(3, 2, 1)
    plt.imshow(one_image, cmap="gray")  # raw data is grayscale images
    plt.axis("off")
    #print(f"Image: {one_image}, Dimensions: {one_image.size}")  # Print dimensions

    img_final = one_image.reshape(-1, 160, 160, 1).astype('float32') / 255.0

    # Predict the fingerprint features
    predictions = model.predict(img_final)
    print("the fingerprint is predicted as person 0000" + str(np.argmax( predictions)) + ' with  ' + str(np.max(predictions)))
    

In [ ]:
# Your code here...

# Exercise 2: Using a pre-trained model using Keras
We can utilize a pre-trained image model to make it easier for our model to learn our dataset. VGG is an classic image model that was trained to classify images during the infamous ImageNet Challenge 2014. What we will do is **transfer** this knowledge from the VGG model to our model by **fine-tuning** the last few layers to our data. 

Prepare the images. this time we will not use grayscale b/c this model has been trained on color images and the color could have additional information. 
- load and size the images
- convert to a numpy array
- use RGB (instead of converting to grayscale)
- Normalize

In [ ]:
def load_and_preprocess_images_color(image_paths):
    '''
     Load and preprocess grayscale fingerprint images for input into a CNN model. 
     '''
    images = []
    labels = []
    for person_id in os.listdir(image_paths):
        person_path = os.path.join(image_paths, person_id)
        if os.path.isdir(person_path):
            for img_file in os.listdir(person_path):
                img_path = os.path.join(person_path, img_file)
                # Load the image
                img = image.load_img(img_path, target_size=(160, 160), color_mode="grayscale")
                # Convert the image to a NumPy array
                img_array = image.img_to_array(img)
                img_array = np.repeat(img_array, 3, axis=-1) / 255.0 
                
                images.append(img_array)
                labels.append(int(person_id))  

    return np.array(images), np.array(labels)

In [ ]:
# Load and process image data for input into CNN

## Use a VGG pre-trained model
### Download model weights
- DownLoad VGG16 weights without the top layer

We exclude the fully connected layers by setting **include_top=False** because we want to replace (**fine-tune**) this with our task (data and 10 classes/labels). 


In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(160, 160, 3))

### Initialize base model
#### Freezing layers
If we use the default weights when adding our layer, the first iteration will be very inaccurate which will create a large gradient. 
Remember, we are using backpropagation which will send these huge errors back to the lower level (nice pre-trained layers) which will mess it all up. 

To keep these lower-level features preserved, we can freeze the base model layers, until our new layers have become more stable. 

In [ ]:
# Freeze the base layers
base_model.trainable = False

In [ ]:
# Build the model architecture that will be specific to our data and task

# Compile the model

### Overfitting
Reduce the overfitting by implementing **EarlyStopping**

In [ ]:
# Early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

## Train the model 
- Use the pre-trained base_model weights initialized above and fit it to our training/validation data.

Notice where the model starts this time in terms of accuracy, and how/if it improves w/ each epoch

In [ ]:
# Train the model


In [ ]:
# Evaluate on the test set

Fit the model again

In [ ]:
# Train the model

In [ ]:
# Evaluate the model on the test set

# Evaluate on the test set

Now fit the model again and choose:
- number of **epochs**
- **batch_size**

Notice what is happening to the accuracy and loss for the train and validation sets as the model continues to train. 

## Saving your model (optional)
Saving a model is straightforward, it can be done with the *pickle* library, or the *h5* library. 

In [ ]:
# Save the model 
model_pre_trained.save('fingerprint_model.keras')

## Loading your model 
If you want to load the model you trained, use the code below

In [ ]:
# Load the entire model
model = load_model('fingerprint_model.keras')